In [1]:
!git clone https://github.com/Team-TUD/CTAB-GAN-Plus
import sys
sys.path.append('./CTAB-GAN-Plus')
from model.ctabgan import CTABGAN

fatal: destination path 'CTAB-GAN-Plus' already exists and is not an empty directory.


In [2]:
pip install sdv

In [3]:
pip install ucimlrepo

In [4]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality

from model.ctabgan import CTABGAN

# Load Adult dataset
adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

data = pd.concat([X, y], axis=1)

target_col = y.columns[0]

# Clean missing markers and sample data
data = data.replace("?", np.nan)
n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

# Fill missing values
numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = data.select_dtypes(include=["object", "category"]).columns

for col in numeric_cols:
    data[col] = data[col].fillna(data[col].mean())

for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

# Encode categorical columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

# Prepare features, target, and metadata
X = data.drop(columns=[target_col])
y = data[target_col]

processed_data = pd.concat([X, y], axis=1)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

# Set experiment constants and seeds
N_SAMPLES = 10000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Initialize result containers
scores = {}
synthetic_datasets = {}
quality_results = []


In [5]:
# Single run

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Train/test split without leakage

train_real, test_real = train_test_split(
    processed_data,
    test_size=TEST_SIZE,
    stratify=processed_data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# CTABGAN

try:
    data_path = "adult_train.csv"
    train_real.to_csv(data_path, index=False)

    categorical_columns = [
        col for col in train_real.columns
        if col != target_col and col in label_encoders
    ]

    ctabgan = CTABGAN(
        raw_csv_path=data_path,
        categorical_columns=categorical_columns + [target_col],
        log_columns=[],
        mixed_columns={},
        integer_columns=[],
        problem_type={"Classification": target_col}
    )

    ctabgan.fit()

    synthetic_ctabgan = ctabgan.data_prep.inverse_prep(
        ctabgan.synthesizer.sample(N_SAMPLES)
    )

    # Ensure the target column in CTABGAN synthetic data is integer-encoded (0 or 1)
    # consistent with the real_data. The `label_encoders` dictionary from previous cells
    # contains the LabelEncoder fitted on the original string labels.
    if target_col in synthetic_ctabgan.columns:
        # Convert to numeric, coercing errors to NaN. This handles object dtype containing numeric strings.
        synthetic_ctabgan[target_col] = pd.to_numeric(synthetic_ctabgan[target_col], errors='coerce')

        # Fill any NaNs that might have been introduced by coercion
        # Filling with the mode of the real target column for consistency.
        real_target_mode = train_real[target_col].mode()[0]
        synthetic_ctabgan[target_col] = synthetic_ctabgan[target_col].fillna(real_target_mode)

        # Convert to integer type
        synthetic_ctabgan[target_col] = synthetic_ctabgan[target_col].astype(int)

        # Ensure values are strictly 0 or 1 by clipping them
        synthetic_ctabgan[target_col] = synthetic_ctabgan[target_col].clip(0, 1)

    synthetic_datasets["CTABGAN"] = synthetic_ctabgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_ctabgan,
        metadata=train_metadata
    )

    score = quality.get_score()

    scores["CTABGAN"] = score

    print("CTABGAN:", round(score, 4))

except Exception as e:
    print("CTABGAN Failed:", e)


================ SINGLE RUN ================


100%|██████████| 150/150 [01:06<00:00,  2.24it/s]


Finished training in 68.98992204666138  seconds.
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 35.89it/s]|
Column Shapes Score: 34.49%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 175.07it/s]|
Column Pair Trends Score: 0.0%

Overall Score (Average): 17.25%

CTABGAN: 0.1725


In [6]:
# WGAN-GP

try:

    import traceback

    data_wgan = train_real.copy()

    encoder = LabelEncoder()
    data_wgan[target_col] = encoder.fit_transform(data_wgan[target_col])

    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(data_wgan)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    real_tensor = torch.tensor(
        scaled_data,
        dtype=torch.float32
    )

    batch_size = 64
    latent_dim = 64
    data_dim = real_tensor.shape[1]

    loader = torch.utils.data.DataLoader(
        real_tensor,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False
    )

    class Generator(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(latent_dim, 128),
                nn.LayerNorm(128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 256),
                nn.LayerNorm(256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, data_dim)
            )

        def forward(self, z):
            return self.model(z)

    class Critic(nn.Module):
        def __init__(self):
            super().__init__()

            self.model = nn.Sequential(
                nn.Linear(data_dim, 256),
                nn.LeakyReLU(0.2),

                nn.Linear(256, 128),
                nn.LeakyReLU(0.2),

                nn.Linear(128, 1)
            )

        def forward(self, x):
            return self.model(x)

    generator = Generator().to(device)
    critic = Critic().to(device)

    optimizer_G = optim.Adam(
        generator.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    optimizer_C = optim.Adam(
        critic.parameters(),
        lr=0.0001,
        betas=(0.5, 0.9)
    )

    def gradient_penalty(critic, real_samples, fake_samples):

        alpha = torch.rand(real_samples.size(0), 1, device=device)
        alpha = alpha.expand_as(real_samples)

        interpolates = (
            alpha * real_samples +
            (1 - alpha) * fake_samples
        ).requires_grad_(True)

        critic_interpolates = critic(interpolates)

        gradients = torch.autograd.grad(
            outputs=critic_interpolates,
            inputs=interpolates,
            grad_outputs=torch.ones_like(critic_interpolates),
            create_graph=True,
            retain_graph=True
        )[0]

        gradients = gradients.view(gradients.size(0), -1)

        return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

    for epoch in range(100):

        for real_batch in loader:

            real_batch = real_batch.to(device)

            for _ in range(5):

                z = torch.randn(
                    real_batch.size(0),
                    latent_dim,
                    device=device
                )

                fake_batch = generator(z).detach()

                critic_real = critic(real_batch).mean()
                critic_fake = critic(fake_batch).mean()

                gp = gradient_penalty(
                    critic,
                    real_batch,
                    fake_batch
                )

                critic_loss = (
                    critic_fake
                    - critic_real
                    + 10 * gp
                )

                optimizer_C.zero_grad()
                critic_loss.backward()
                optimizer_C.step()

            z = torch.randn(
                real_batch.size(0),
                latent_dim,
                device=device
            )

            fake = generator(z)

            generator_loss = -critic(fake).mean()

            optimizer_G.zero_grad()
            generator_loss.backward()
            optimizer_G.step()

    generator.eval()

    with torch.no_grad():

        z = torch.randn(
            N_SAMPLES,
            latent_dim,
            device=device
        )

        synthetic_scaled = generator(z).cpu().numpy()

    synthetic = scaler.inverse_transform(synthetic_scaled)

    synthetic_wgan = pd.DataFrame(
        synthetic,
        columns=data_wgan.columns
    )

    synthetic_wgan[target_col] = (
        synthetic_wgan[target_col]
        .round()
        .clip(0, 1)
        .astype(int)
    )

    # Removed the following line as it converts target column back to strings,
    # which causes issues with evaluate_quality when compared to integer-encoded real data.
    # synthetic_wgan[target_col] = encoder.inverse_transform(
    #     synthetic_wgan[target_col]
    # )

    synthetic_datasets["WGAN_GP"] = synthetic_wgan.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_wgan,
        metadata=train_metadata
    )

    scores["WGAN_GP"] = quality.get_score()

    print("WGAN_GP:", round(scores["WGAN_GP"], 4))

    del generator
    del critic
    del real_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("WGAN_GP Failed:")
    traceback.print_exc()

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:12<00:00,  1.18it/s]|
Column Shapes Score: 45.14%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 239.27it/s]|
Column Pair Trends Score: -0.0%

Overall Score (Average): 22.57%

WGAN_GP: 0.2257


In [7]:
# SDV MODELS

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 44.72it/s]|
Column Shapes Score: 84.6%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 194.59it/s]|
Column Pair Trends Score: 70.51%

Overall Score (Average): 77.55%

CTGAN: 0.7755
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 83.60it/s]|
Column Shapes Score: 88.21%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 266.79it/s]|
Column Pair Trends Score: 52.55%

Overall Score (Average): 70.38%

CopulaGAN: 0.7038
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 58.23it/s]|
Column Shapes Score: 79.08%

(2/2) Evaluating Column Pair Trends: |██████████| 105/105 [00:00<00:00, 209.93it/s]|
Column Pair Trends Score: 62.17%

Overall Score (Average): 70.62%

TVAE: 0.7062
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 15/15 [00:00<00:00, 46.21it/s]|
Column Shapes

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=None
):
    if seeds is None:
        seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

    results = []

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            # Extract X and y from the full training DataFrame for the current seed
            X_train_full = train_df.drop(columns=[label_col])
            y_train_full = train_df[label_col]

            # Extract X and y from the full testing DataFrame for the current seed
            X_test_full = test_df.drop(columns=[label_col])
            y_test_full = test_df[label_col]

            # Determine if stratification is possible for y_train_full
            stratify_y_train_full = y_train_full if y_train_full.value_counts().min() >= 2 else None
            if stratify_y_train_full is None:
                print(
                    f"Warning: Cannot stratify training data for model {name} with seed {seed} "
                    f"due to a class with <2 samples in `train_df`'s target column. "
                    f"Proceeding without stratification for this split."
                )

            # Perform the first train-test split (from train_df to get actual training set for classifier)
            X_train_split, _, y_train_split, _ = train_test_split(
                X_train_full,
                y_train_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_y_train_full
            )

            # Determine if stratification is possible for y_test_full
            # This is usually for real data and should be fine, but included for robustness.
            stratify_y_test_full = y_test_full if y_test_full.value_counts().min() >= 2 else None
            if stratify_y_test_full is None:
                print(
                    f"Warning: Cannot stratify testing data for model {name} with seed {seed} "
                    f"due to a class with <2 samples in `test_df`'s target column. "
                    f"Proceeding without stratification for this split."
                )

            # Perform the second train-test split (from test_df to get actual testing set for classifier)
            _, X_test_split, _, y_test_split = train_test_split(
                X_test_full,
                y_test_full,
                test_size=test_size,
                random_state=seed,
                stratify=stratify_y_test_full
            )

            scaler = StandardScaler().fit(X_train_split)

            X_train_s = scaler.transform(X_train_split)
            X_test_s = scaler.transform(X_test_split)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            clf.fit(X_train_s, y_train_split)

            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(
                accuracy_score(y_test_split, y_pred)
            )

            f1_scores.append(
                f1_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

            precision_scores.append(
                precision_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

            recall_scores.append(
                recall_score(
                    y_test_split,
                    y_pred,
                    average="weighted",
                    zero_division=0
                )
            )

        acc_mean = np.mean(accuracy_scores)
        acc_std = np.std(accuracy_scores, ddof=1)

        f1_mean = np.mean(f1_scores)
        f1_std = np.std(f1_scores, ddof=1)

        prec_mean = np.mean(precision_scores)
        prec_std = np.std(precision_scores, ddof=1)

        rec_mean = np.mean(recall_scores)
        rec_std = np.std(recall_scores, ddof=1)

        results.append({
            "Model": name,

            "Accuracy Mean": acc_mean,
            "Accuracy Std": acc_std,
            "F1 Mean": f1_mean,
            "F1 Std": f1_std,
            "Precision Mean": prec_mean,
            "Precision Std": prec_std,
            "Recall Mean": rec_mean,
            "Recall Std": rec_std,

            "Accuracy \u00b1 SD": f"{acc_mean:.4f} \u00b1 {acc_std:.4f}",
            "F1 \u00b1 SD": f"{f1_mean:.4f} \u00b1 {f1_std:.4f}",
            "Precision \u00b1 SD": f"{prec_mean:.4f} \u00b1 {prec_std:.4f}",
            "Recall \u00b1 SD": f"{rec_mean:.4f} \u00b1 {rec_std:.4f}",

            "Accuracy (Mean\u00b1Std)": f"{acc_mean:.4f} \u00b1 {acc_std:.4f}",
            "F1 (Mean\u00b1Std)": f"{f1_mean:.4f} \u00b1 {f1_std:.4f}",
            "Precision (Mean\u00b1Std)": f"{prec_mean:.4f} \u00b1 {prec_std:.4f}",
            "Recall (Mean\u00b1Std)": f"{rec_mean:.4f} \u00b1 {rec_std:.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False
    )

In [10]:
# Load Adult dataset
adult = fetch_ucirepo(id=2)

X = adult.data.features
y = adult.data.targets

data = pd.concat([X, y], axis=1)

target_col = y.columns[0]

# Clean missing markers
data = data.replace("?", np.nan)

# Clean income target into two classes only
data[target_col] = (
    data[target_col]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.strip()
)

print(data[target_col].value_counts())

# Take 1000 real samples
n_samples = min(1000, len(data))
data = data.sample(n=n_samples, random_state=42).reset_index(drop=True)

# Fill missing values
numeric_cols = data.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = data.select_dtypes(include=["object", "category"]).columns

for col in numeric_cols:
    data[col] = data[col].fillna(data[col].mean())

for col in categorical_cols:
    data[col] = data[col].fillna(data[col].mode()[0])

# Encode categorical columns
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))
    label_encoders[col] = le

# Prepare processed dataset
X = data.drop(columns=[target_col])
y = data[target_col]

processed_data = pd.concat([X, y], axis=1)

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(processed_data)

print("\nEncoded income classes:")
print(processed_data[target_col].value_counts())
print(label_encoders[target_col].classes_)


income
<=50K    37155
>50K     11687
Name: count, dtype: int64

Encoded income classes:
income
0    775
1    225
Name: count, dtype: int64
['<=50K' '>50K']


In [11]:
# TRTR evaluation for Adult income dataset

print("--- Starting TRTR Evaluation (Train Real, Test Real) ---")

trtr_results = []

SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

X = processed_data.drop(columns=[target_col])
y = processed_data[target_col]

print("Target column:", target_col)
print("Target classes:")
print(y.value_counts())

for model_name, model in models.items():

    accuracy_scores = []
    f1_scores = []
    precision_scores = []
    recall_scores = []

    print(f"Running {model_name}...")

    for seed in SEEDS:

        X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            stratify=y,
            random_state=seed
        )

        clf = clone(model)

        if hasattr(clf, "random_state"):
            clf.set_params(random_state=seed)

        clf.fit(X_train_real, y_train_real)

        y_pred = clf.predict(X_test_real)

        accuracy_scores.append(
            accuracy_score(y_test_real, y_pred)
        )

        f1_scores.append(
            f1_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        precision_scores.append(
            precision_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

        recall_scores.append(
            recall_score(
                y_test_real,
                y_pred,
                average="weighted",
                zero_division=0
            )
        )

    acc_mean = np.mean(accuracy_scores)
    acc_std = np.std(accuracy_scores, ddof=1)

    f1_mean = np.mean(f1_scores)
    f1_std = np.std(f1_scores, ddof=1)

    prec_mean = np.mean(precision_scores)
    prec_std = np.std(precision_scores, ddof=1)

    rec_mean = np.mean(recall_scores)
    rec_std = np.std(recall_scores, ddof=1)

    trtr_results.append({
        "Model": model_name,

        "Accuracy Mean_TRTR": acc_mean,
        "Accuracy Std_TRTR": acc_std,
        "F1 Mean_TRTR": f1_mean,
        "F1 Std_TRTR": f1_std,
        "Precision Mean_TRTR": prec_mean,
        "Precision Std_TRTR": prec_std,
        "Recall Mean_TRTR": rec_mean,
        "Recall Std_TRTR": rec_std,

        "Accuracy (Mean±Std)_TRTR": f"{acc_mean:.4f} ± {acc_std:.4f}",
        "F1 (Mean±Std)_TRTR": f"{f1_mean:.4f} ± {f1_std:.4f}",
        "Precision (Mean±Std)_TRTR": f"{prec_mean:.4f} ± {prec_std:.4f}",
        "Recall (Mean±Std)_TRTR": f"{rec_mean:.4f} ± {rec_std:.4f}"
    })

trtr_results_df = pd.DataFrame(trtr_results).sort_values(
    by="Accuracy Mean_TRTR",
    ascending=False
)

display(
    trtr_results_df[
        [
            "Model",
            "Accuracy (Mean±Std)_TRTR",
            "F1 (Mean±Std)_TRTR",
            "Precision (Mean±Std)_TRTR",
            "Recall (Mean±Std)_TRTR"
        ]
    ]
)

--- Starting TRTR Evaluation (Train Real, Test Real) ---
Target column: income
Target classes:
income
0    775
1    225
Name: count, dtype: int64
Running LogReg...
Running SVM-RBF...
Running KNN...
Running NaiveBayes...
Running DecisionTree...
Running RandomForest...
Running ExtraTrees...
Running GradientBoost...
Running AdaBoost...
Running MLP...


,Model,Accuracy (Mean±Std)_TRTR,F1 (Mean±Std)_TRTR,Precision (Mean±Std)_TRTR,Recall (Mean±Std)_TRTR
5,RandomForest,0.8375 ± 0.0265,0.8293 ± 0.0292,0.8280 ± 0.0305,0.8375 ± 0.0265
8,AdaBoost,0.8370 ± 0.0234,0.8271 ± 0.0270,0.8275 ± 0.0280,0.8370 ± 0.0234
7,GradientBoost,0.8330 ± 0.0250,0.8246 ± 0.0259,0.8241 ± 0.0269,0.8330 ± 0.0250
6,ExtraTrees,0.8240 ± 0.0247,0.8177 ± 0.0258,0.8164 ± 0.0265,0.8240 ± 0.0247
0,LogReg,0.8065 ± 0.0210,0.7690 ± 0.0291,0.7884 ± 0.0370,0.8065 ± 0.0210
3,NaiveBayes,0.8040 ± 0.0249,0.7768 ± 0.0291,0.7829 ± 0.0389,0.8040 ± 0.0249
4,DecisionTree,0.7845 ± 0.0290,0.7857 ± 0.0286,0.7883 ± 0.0290,0.7845 ± 0.0290
1,SVM-RBF,0.7760 ± 0.0021,0.6791 ± 0.0049,0.6462 ± 0.0961,0.7760 ± 0.0021
2,KNN,0.7700 ± 0.0207,0.7229 ± 0.0247,0.7220 ± 0.0390,0.7700 ± 0.0207
9,MLP,0.5795 ± 0.3054,0.4985 ± 0.3587,0.5086 ± 0.3949,0.5795 ± 0.3054


In [12]:
import pandas as pd

label_col = target_col   # Adult dataset target column: income

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "WGAN_GP",
    "CTABGAN"
]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

real_data = processed_data.copy()

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=real_data,
    test_df=real_data,
    label="income",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy \u00b1 SD",
            "F1 \u00b1 SD",
            "Precision \u00b1 SD",
            "Recall \u00b1 SD"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    if synth_name not in synthetic_datasets:
        print(f"{synth_name} not found in synthetic_datasets. Skipping.")
        continue

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name].copy()

    # Ensure the target column in the synthetic data is integer-encoded (0 or 1)
    # consistent with the real_data. The `label_encoders` dictionary from previous cells
    # contains the LabelEncoder fitted on the original string labels.
    if label_col in synthetic_train_df.columns:
        if pd.api.types.is_object_dtype(synthetic_train_df[label_col]):
            # If the column contains string labels, transform them to integers
            synthetic_train_df[label_col] = label_encoders[label_col].transform(synthetic_train_df[label_col])
        elif not pd.api.types.is_integer_dtype(synthetic_train_df[label_col]):
            # If it's numeric but not integer (e.g., float), convert to integer
            synthetic_train_df[label_col] = synthetic_train_df[label_col].astype(int)

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=real_data,
        label="income",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy \u00b1 SD",
                "F1 \u00b1 SD",
                "Precision \u00b1 SD",
                "Recall \u00b1 SD"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=('_TRTR', '_TSTR')
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy \u00b1 SD_TRTR",
                "Accuracy \u00b1 SD_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
5,RandomForest,0.8385 ± 0.0258,0.8306 ± 0.0285,0.8293 ± 0.0298,0.8385 ± 0.0258
8,AdaBoost,0.8370 ± 0.0234,0.8271 ± 0.0270,0.8275 ± 0.0280,0.8370 ± 0.0234
1,SVM-RBF,0.8335 ± 0.0143,0.8149 ± 0.0186,0.8237 ± 0.0183,0.8335 ± 0.0143
7,GradientBoost,0.8325 ± 0.0250,0.8242 ± 0.0259,0.8235 ± 0.0270,0.8325 ± 0.0250
0,LogReg,0.8280 ± 0.0218,0.8104 ± 0.0263,0.8146 ± 0.0279,0.8280 ± 0.0218
6,ExtraTrees,0.8240 ± 0.0247,0.8177 ± 0.0258,0.8164 ± 0.0265,0.8240 ± 0.0247
3,NaiveBayes,0.8090 ± 0.0258,0.7857 ± 0.0299,0.7898 ± 0.0386,0.8090 ± 0.0258
9,MLP,0.8075 ± 0.0255,0.8070 ± 0.0238,0.8083 ± 0.0222,0.8075 ± 0.0255
2,KNN,0.8070 ± 0.0225,0.8009 ± 0.0251,0.7978 ± 0.0266,0.8070 ± 0.0225
4,DecisionTree,0.7840 ± 0.0291,0.7853 ± 0.0286,0.7881 ± 0.0289,0.7840 ± 0.0291


CTGAN - TSTR


,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
0,LogReg,0.7750 ± 0.0000,0.6768 ± 0.0000,0.6006 ± 0.0000,0.7750 ± 0.0000
8,AdaBoost,0.7750 ± 0.0000,0.6768 ± 0.0000,0.6006 ± 0.0000,0.7750 ± 0.0000
1,SVM-RBF,0.7745 ± 0.0016,0.6765 ± 0.0008,0.6005 ± 0.0003,0.7745 ± 0.0016
7,GradientBoost,0.7570 ± 0.0086,0.6695 ± 0.0070,0.6101 ± 0.0381,0.7570 ± 0.0086
3,NaiveBayes,0.7410 ± 0.0154,0.6703 ± 0.0200,0.6680 ± 0.0993,0.7410 ± 0.0154
5,RandomForest,0.6910 ± 0.0200,0.6549 ± 0.0194,0.6292 ± 0.0237,0.6910 ± 0.0200
6,ExtraTrees,0.6810 ± 0.0165,0.6505 ± 0.0157,0.6285 ± 0.0202,0.6810 ± 0.0165
9,MLP,0.6305 ± 0.0364,0.6303 ± 0.0284,0.6322 ± 0.0264,0.6305 ± 0.0364
2,KNN,0.6025 ± 0.0307,0.6219 ± 0.0286,0.6436 ± 0.0277,0.6025 ± 0.0307
4,DecisionTree,0.4145 ± 0.0519,0.4955 ± 0.0508,0.6537 ± 0.0441,0.4145 ± 0.0519


CTGAN - TRTR vs TSTR


...

,Model,Accuracy ± SD,F1 ± SD,Precision ± SD,Recall ± SD
0,LogReg,0.7710 ± 0.0057,0.6792 ± 0.0060,0.6572 ± 0.0765,0.7710 ± 0.0057
8,AdaBoost,0.7420 ± 0.0470,0.6962 ± 0.0362,0.6759 ± 0.0694,0.7420 ± 0.0470
3,NaiveBayes,0.7165 ± 0.0329,0.6895 ± 0.0365,0.6730 ± 0.0450,0.7165 ± 0.0329
1,SVM-RBF,0.7135 ± 0.0210,0.6664 ± 0.0175,0.6374 ± 0.0224,0.7135 ± 0.0210
6,ExtraTrees,0.5995 ± 0.0444,0.6146 ± 0.0363,0.6342 ± 0.0271,0.5995 ± 0.0444
9,MLP,0.5445 ± 0.0535,0.5747 ± 0.0444,0.6234 ± 0.0298,0.5445 ± 0.0535
2,KNN,0.5395 ± 0.0468,0.5736 ± 0.0419,0.6330 ± 0.0372,0.5395 ± 0.0468
4,DecisionTree,0.4880 ± 0.1082,0.5199 ± 0.1090,0.6546 ± 0.0686,0.4880 ± 0.1082
5,RandomForest,0.4635 ± 0.0940,0.4903 ± 0.1094,0.6695 ± 0.0392,0.4635 ± 0.0940
7,GradientBoost,0.2820 ± 0.1230,0.2162 ± 0.1686,0.4345 ± 0.2875,0.2820 ± 0.1230


CTABGAN - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy ± SD_TRTR,Accuracy ± SD_TSTR
0,CTABGAN,RandomForest,0.3750,0.340243,0.159813,0.3750,0.8385 ± 0.0258,0.4635 ± 0.0940
1,CTABGAN,AdaBoost,0.0950,0.130968,0.151574,0.0950,0.8370 ± 0.0234,0.7420 ± 0.0470
2,CTABGAN,SVM-RBF,0.1200,0.148502,0.186345,0.1200,0.8335 ± 0.0143,0.7135 ± 0.0210
3,CTABGAN,GradientBoost,0.5505,0.608013,0.389091,0.5505,0.8325 ± 0.0250,0.2820 ± 0.1230
4,CTABGAN,LogReg,0.0570,0.131176,0.157373,0.0570,0.8280 ± 0.0218,0.7710 ± 0.0057
5,CTABGAN,ExtraTrees,0.2245,0.203071,0.182252,0.2245,0.8240 ± 0.0247,0.5995 ± 0.0444
6,CTABGAN,NaiveBayes,0.0925,0.096224,0.116753,0.0925,0.8090 ± 0.0258,0.7165 ± 0.0329
7,CTABGAN,MLP,0.2630,0.232231,0.184980,0.2630,0.8075 ± 0.0255,0.5445 ± 0.0535
8,CTABGAN,KNN,0.2675,0.227306,0.164850,0.2675,0.8070 ± 0.0225,0.5395 ± 0.0468
9,CTABGAN,DecisionTree,0.2960,0.265392,0.133450,0.2960,0.7840 ± 0.0291,0.4880 ± 0.1082


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
1,CTGAN,0.13590,0.168086,0.185184,0.13590
4,TVAE,0.15035,0.153468,0.148753,0.15035
3,GaussianCopula,0.15610,0.184346,0.204953,0.15610
2,CopulaGAN,0.15720,0.173339,0.174623,0.15720
0,CTABGAN,0.23410,0.238313,0.182648,0.23410
5,WGAN_GP,0.47025,0.500403,0.041457,0.47025


In [14]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
